In [27]:
import pandas as pd
import pandas as pd
import numpy as np
from pathlib import Path
import tqdm

df_og = pd.read_csv("../labels/gt/source_files_rsna/annotations.csv")


def filter_fun(row):
    if row["w"] > 250 and row["h"] > 250:
        return False
    return True


df_og["keep"] = df_og.apply(filter_fun, axis=1)
df_og_filtered = df_og[df_og["keep"] == True]
df_og_filtered = df_og_filtered.drop(columns=["keep"])
# to csv
df_og_filtered.to_csv(
    "../labels/gt/source_files_rsna/annotations_filt.csv",
    index=False,
)
df_og_filtered

,seriesuid,coordX,coordY,coordZ,w,h,d,lesion,volume,min_axis,maj_axis
0,1.2.826.0.1.3680043.8.498.10005158603912009425...,283.0,275.0,162.0,11,11,1,aneurysm,121.0,0.000000,14.142136
1,1.2.826.0.1.3680043.8.498.10022796280698534221...,223.0,384.5,153.0,11,12,1,aneurysm,132.0,1.000000,15.438048
2,1.2.826.0.1.3680043.8.498.10030095840917973694...,198.5,270.5,113.5,10,10,2,aneurysm,200.0,2.236068,12.845233
3,1.2.826.0.1.3680043.8.498.10035643165968342618...,274.5,338.5,112.0,12,12,1,aneurysm,144.0,0.000000,15.438048
4,1.2.826.0.1.3680043.8.498.10035643165968342618...,264.0,340.0,129.5,11,11,2,aneurysm,242.0,2.236068,14.142136
...,...,...,...,...,...,...,...,...,...,...,...
1142,1.2.826.0.1.3680043.8.498.99887675554378211308...,100.0,211.5,115.5,7,6,2,aneurysm,84.0,2.236068,8.944272
1143,1.2.826.0.1.3680043.8.498.99887675554378211308...,133.0,193.5,137.5,7,6,2,aneurysm,84.0,2.236068,8.944272
1144,1.2.826.0.1.3680043.8.498.99892390884723813599...,172.5,294.5,175.0,10,10,3,aneurysm,300.0,3.651484,12.845233
1145,1.2.826.0.1.3680043.8.498.99892390884723813599...,355.0,309.0,200.0,9,9,3,aneurysm,243.0,3.651484,11.547005


In [28]:
def overlaps(z, y, x, z_pred, y_pred, x_pred, d, h, w):
    z_pred_0 = z_pred - d / 2
    z_pred_1 = z_pred + d / 2
    y_pred_0 = y_pred - h / 2
    y_pred_1 = y_pred + h / 2
    x_pred_0 = x_pred - w / 2
    x_pred_1 = x_pred + w / 2
    # verify if z,y,x is inside the predicted box
    if (
        (z >= z_pred_0 and z <= z_pred_1)
        and (y >= y_pred_0 and y <= y_pred_1)
        and (x >= x_pred_0 and x <= x_pred_1)
    ):

        return True

    else:
        return False

In [29]:
path_predictions = [
    "../results/cta_rsna_ane/.old/cnn_4l_input_edt_amp_ogopt/inference_final/predict.csv",
    "../results/cta_rsna_ane/.old/aug_cnnfa_4l_short_input_edt_fp_strict_fix/inference_final/predict.csv",
    "../results/cta_rsna_ane/.old/opaug_cnn_4l_short_input_edt/inference_final/predict.csv",
]
path_annot = (
    "/scratch/ceballosarroyo.a/aneurysm/mm_datasets/cta_rsna_ane/annotations.csv"
)
df_annot = df_og_filtered
headers = [
    "idx",
    "seriesuid",
    "coordZ",
    "coordY",
    "coordX",
    "d",
    "h",
    "w",
    "probability",
]


values = []
has_pred = []
for i, row in tqdm.tqdm(df_annot.iterrows()):
    seriesuid = row["seriesuid"]
    z, y, x = row["coordZ"], row["coordY"], row["coordX"]

    for path_pred in path_predictions:
        df_pred = pd.read_csv(path_pred)
        pred_row = df_pred[df_pred["seriesuid"] == seriesuid]
        # check
        z_pred, y_pred, x_pred, d, h, w = pred_row[
            ["coordZ", "coordY", "coordX", "d", "h", "w"]
        ].values[0]
        # check if there is overlap
        over = overlaps(z, y, x, z_pred, y_pred, x_pred, d, h, w)
        if over:
            values.append(
                [
                    i,
                    seriesuid,
                    z_pred,
                    y_pred,
                    x_pred,
                    d,
                    h,
                    w,
                    pred_row["probability"].values[0],
                ]
            )
            has_pred.append(i)

1065it [01:42, 10.40it/s]


In [30]:
has_pred = set(has_pred)
print(f"Number of annotations with predictions: {len(has_pred)}/{len(df_annot)}")

Number of annotations with predictions: 843/1065


In [38]:
df_new_headers = df_annot.columns

rows = []

all_d = [v[5] for v in values]
all_h = [v[6] for v in values]
all_w = [v[7] for v in values]

count = 0
for idx, row in df_annot.iterrows():
    vals = [v for v in values if v[0] == idx]
    sample_d = [v[5] for v in vals]
    sample_h = [v[6] for v in vals]
    sample_w = [v[7] for v in vals]
    proba = [v[8] for v in vals]

    for i_p, p in enumerate(proba):
        if p < 0.5:
            sample_d.pop(i_p)
            sample_h.pop(i_p)
            sample_w.pop(i_p)

    if len(sample_d) > 0:
        mean_d = np.mean(sample_d)
        mean_h = np.mean(sample_h)
        mean_w = np.mean(sample_w)
        count += 1
    else:
        mean_d = np.mean(all_d)
        mean_h = np.mean(all_h)
        mean_w = np.mean(all_w)

    volume = mean_d * mean_h * mean_w
    major_axis = max(mean_d, mean_h, mean_w)
    minor_axis = min(mean_d, mean_h, mean_w)

    new_row = [
        row["seriesuid"],
        row["coordX"],
        row["coordY"],
        row["coordZ"],
        mean_d,
        mean_h,
        mean_w,
        "aneurysm",
        volume,
        major_axis,
        minor_axis,
    ]

    rows.append(new_row)


df_annot_with_sizes = pd.DataFrame(
    rows,
    columns=[
        "seriesuid",
        "coordX",
        "coordY",
        "coordZ",
        "d",
        "h",
        "w",
        "label",
        "volume",
        "major_axis",
        "minor_axis",
    ],
)

In [39]:
# get_list_of_every uid

files = list(df_annot["seriesuid"].unique().tolist())

In [32]:
df_annot_with_sizes.to_csv(
    "../labels/gt/source_files_rsna/annotations_auto.csv", index=False
)

In [ ]:
df_manual = pd.read_csv("../labels/gt/source_files_rsna/annotations_manual.csv")

# for each in df_annot_with_sizes, check if something with the same coordX, coordY, and coordZ exists in df_manual

print(df_annot_with_sizes.shape)
df_annot_with_sizes["keep"] = True

for i, row in df_annot_with_sizes.iterrows():
    row_manual = df_manual[
        (df_manual["coordX"] == row["coordX"])
        & (df_manual["coordY"] == row["coordY"])
        & (df_manual["coordZ"] == row["coordZ"])
    ]
    if len(row_manual) > 0:
        df_annot_with_sizes.at[i, "keep"] = False
# print the list of annotation sin manual wit keep = False



df_annot_with_sizes = df_annot_with_sizes[df_annot_with_sizes["keep"] == True]
df_annot_with_sizes = df_annot_with_sizes.drop(columns=["keep"])
print(df_annot_with_sizes.shape)

(821, 11)
[]
(821, 11)


In [34]:
path_data = Path("/scratch/ceballosarroyo.a/aneurysm/mm_datasets/cta_rsna_ane/crop_0.4")
# get unique filenames

files = list(path_data.glob("**/*.nii.gz"))

# get only the names

file_names = [f.name for f in files]

# only get the filenames in the list
df_annot_with_sizes = df_annot_with_sizes[
    df_annot_with_sizes["seriesuid"].isin(file_names)
]
df_manual = df_manual[df_manual["seriesuid"].isin(file_names)]

In [35]:
df_annot_with_sizes.shape, df_manual.shape

((821, 11), (208, 17))

In [36]:
df_manual["coordX"] = df_manual["new_x"]
df_manual["coordY"] = df_manual["new_y"]
df_manual["coordZ"] = df_manual["new_z"]
df_manual["d"] = df_manual["new_d"]
df_manual["h"] = df_manual["new_h"]
df_manual["w"] = df_manual["new_w"]

df_manual = df_manual[df_annot_with_sizes.columns]

df_annot_with_sizes_final = pd.concat(
    [df_annot_with_sizes, df_manual], ignore_index=True
)
df_annot_with_sizes_final.to_csv(
    "../labels/gt/source_files_rsna/annotations_final.csv", index=False
)
print(len(df_annot_with_sizes_final))

1029
